# Course 8: Mathematics and AI
## Session 1: Groups and their actions

**Goal:** build C4, D4 and Sn as concrete objects, check the group axioms by computer, and test which models respect which symmetries.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every Python cell without a NOT RUN label was run once on a CPU (November 2026). Cells labelled NOT RUN need a library, a GPU or a Lean installation. Lean code in this course was written carefully but was NOT compiled here: check it in Lean before you teach it.

### 1. Three groups as matrices
- C4: the four rotations of a square.
- D4: the four rotations and four reflections.
- Sn: all orderings of n things, as permutation matrices.

In [ ]:
import itertools, numpy as np
R = np.array([[0, -1], [1, 0]]); F = np.array([[1, 0], [0, -1]])               # quarter turn, and a reflection
C4 = [np.linalg.matrix_power(R, k) for k in range(4)]
D4 = C4 + [g @ F for g in C4]
def S(n): return [np.eye(n, dtype=int)[list(p)] for p in itertools.permutations(range(n))]
print("sizes: C4", len(C4), "| D4", len(D4), "| S3", len(S(3)), "| S4", len(S(4)))

### 2. Check the axioms
A group needs closure, an identity, inverses and associativity. For a finite set of matrices the computer can check every case.

In [ ]:
def index(G, m): return next(i for i, g in enumerate(G) if np.array_equal(g, m))
def is_group(G):
    n = G[0].shape[0]; I = np.eye(n, dtype=int)
    closure = all(any(np.array_equal(a @ b, g) for g in G) for a in G for b in G)
    identity = any(np.array_equal(g, I) for g in G)
    inverses = all(any(np.array_equal(a @ b, I) for b in G) for a in G)
    return closure and identity and inverses           # associativity is inherited from matrix multiplication
def is_abelian(G): return all(np.array_equal(a @ b, b @ a) for a in G for b in G)

for name, G in [("C4", C4), ("D4", D4), ("S3", S(3))]:
    print(f"{name}: group {is_group(G)} | commutative {is_abelian(G)}")
not_a_group = [C4[0], C4[1]]                             # identity and one quarter turn: not closed
print("identity with one quarter turn:", is_group(not_a_group))
assert is_group(D4) and not is_abelian(D4) and not is_group(not_a_group)

### 3. The multiplication table of D4

In [ ]:
names = ["e", "r", "r2", "r3", "f", "rf", "r2f", "r3f"]
table = [[names[index(D4, a @ b)] for b in D4] for a in D4]
print("     " + " ".join(f"{n:>4s}" for n in names))
for n, row in zip(names, table): print(f"{n:>4s} " + " ".join(f"{x:>4s}" for x in row))
assert all(len(set(row)) == 8 for row in table)          # every row is a rearrangement of the group (Latin square)

### 4. A group action on images
An action lets each group element move a data point. It must satisfy `e · x = x` and `g · (h · x) = (g h) · x`. Here D4 acts on square images by rotating and flipping.

In [ ]:
def act(k, flip, img):                                    # the element r^k f^flip
    return np.rot90(np.fliplr(img) if flip else img, k)
elems = [(k, f) for f in (0, 1) for k in range(4)]        # same order as D4 above
def mul(a, b): i = index(D4, D4[elems.index(a)] @ D4[elems.index(b)]); return elems[i]

rng = np.random.default_rng(0); img = rng.random((8, 8))
ok = all(np.allclose(act(*g, act(*h, img)), act(*mul(g, h), img)) for g in elems for h in elems)
print("g · (h · x) = (gh) · x for all 64 pairs:", ok)
assert ok and np.allclose(act(0, 0, img), img)
orbit = {act(*g, img).tobytes() for g in elems}
sym = np.ones((8, 8)); print("orbit size of a random image:", len(orbit), "| of a blank image:", len({act(*g, sym).tobytes() for g in elems}))

### 5. Invariant and equivariant functions
- Invariant: `f(g · x) = f(x)`. Example: the total brightness.
- Equivariant: `f(g · x) = g · f(x)`. Example: blurring with a symmetric kernel.

In [ ]:
total = lambda x: x.sum()
def blur(x):
    p = np.pad(x, 1, mode="wrap"); return (p[:-2, 1:-1] + p[2:, 1:-1] + p[1:-1, :-2] + p[1:-1, 2:] + p[1:-1, 1:-1]) / 5
def shift_right(x):
    return np.roll(x, 1, axis=1)
for name, f, kind in [("total brightness", total, "invariant"), ("symmetric blur", blur, "equivariant"), ("shift right by one", shift_right, "equivariant")]:
    if kind == "invariant": ok = all(np.isclose(f(act(*g, img)), f(img)) for g in elems)
    else: ok = all(np.allclose(f(act(*g, img)), act(*g, f(img))) for g in elems)
    print(f"{name:20s} is D4-{kind}: {ok}")

Shifting right is equivariant to translations, but not to rotations: rotate first and 'right' points somewhere else.

### 6. Does a trained model respect the symmetry?
A plain classifier trained on upright digits has no reason to. Measure the drop.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
d = load_digits(); Xtr, Xte, ytr, yte = train_test_split(d.images / 16.0, d.target, test_size=0.3, random_state=0)
clf = LogisticRegression(max_iter=2000).fit(Xtr.reshape(len(Xtr), -1), ytr)
for k in range(4):
    acc = clf.score(np.stack([np.rot90(x, k) for x in Xte]).reshape(len(Xte), -1), yte)
    print(f"test digits rotated by {90 * k:3d} degrees: accuracy {acc:.3f}")
# One fix that needs no new architecture: average the model over the group (this makes any model invariant).
def invariant_predict(X): return sum(clf.predict_proba(np.stack([np.rot90(x, k) for x in X]).reshape(len(X), -1)) for k in range(4)).argmax(1)
same = np.array_equal(invariant_predict(Xte), invariant_predict(np.stack([np.rot90(x, 1) for x in Xte])))
print("group-averaged model gives the same answer for rotated inputs:", same)
assert same

Averaging over the group makes the model invariant, but it does not make it accurate: it averages predictions on images the model was never trained to read. Building the symmetry into the layers (Course 6, session 6) does both.

### 7. Permutations and graphs
A GCN layer commutes with the action of Sn on node labels (Course 6, session 1). Check it here for every element of S4.

In [ ]:
A = np.array([[0, 1, 1, 0], [1, 0, 1, 0], [1, 1, 0, 1], [0, 0, 1, 0]], dtype=float); X = rng.normal(size=(4, 3)); W = rng.normal(size=(3, 2))
def gcn(X, A):
    Ah = A + np.eye(4); dg = Ah.sum(1); return np.tanh((Ah / np.sqrt(np.outer(dg, dg))) @ X @ W)
ok = all(np.allclose(gcn(P @ X, P @ A @ P.T), P @ gcn(X, A)) for P in S(4))
print("GCN layer is equivariant to all 24 permutations:", ok); assert ok

### TODO
1. Find all subgroups of D4 by brute force (subsets that pass `is_group`). How many are there?
2. Show that the map D4 -> {+1, -1} given by the determinant is a homomorphism. What is its kernel?
3. Which of these are C4-invariant, C4-equivariant, or neither: the maximum pixel, the transpose of the image, the top-left pixel?